In [1]:
import numpy as np
import cv2
import serial
import time
from tensorflow.keras.models import load_model
import tensorflow as tf


In [2]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()

for port in ports:
    print(port.device)

/dev/cu.debug-console
/dev/cu.Bluetooth-Incoming-Port
/dev/cu.Airdopes301
/dev/cu.soundcoreV20i
/dev/cu.usbserial-0001
/dev/cu.SLAB_USBtoUART


In [3]:
model = tf.keras.models.load_model('trained_model.keras')

/Users/ajayyadav/Downloads/eAgro/eAgro/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 26 variables whereas the saved optimizer has 50 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import serial
import time


esp = serial.Serial('/dev/cu.SLAB_USBtoUART', 115200)
time.sleep(2)

class_name = [
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

image_path = '/Users/ajayyadav/Downloads/eAgro/Dataset/test/TomatoHealthy2.JPG'

img = cv2.imread(image_path)

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

image = tf.keras.preprocessing.image.load_img(
    image_path,
    target_size=(256, 256)
)

input_arr = tf.keras.preprocessing.image.img_to_array(image)
input_arr = np.array([input_arr])

prediction = model.predict(input_arr)

result_index = np.argmax(prediction)

label = class_name[result_index]

confidence = prediction[0][result_index] * 100

print("Disease Name :", label)
print("Confidence Score : {:.2f}%".format(confidence))

if label in ["Tomato___healthy", "Potato___healthy"]:
    print("Healthy plant → Water")
    esp.write(b'WATER\n')

elif label in [
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Target_Spot"
]:
    print("Disease group 1 → Pump1")
    esp.write(b'PEST1\n')

elif label in [
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus"
]:
    print("Disease group 2 → Pump2")
    esp.write(b'PEST2\n')


cv2.putText(
    img,
    f"Disease: {label}",
    (20, 40),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (0, 255, 0),
    2
)

cv2.putText(
    img,
    f"Confidence: {confidence:.2f}%",
    (20, 80),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.5,
    (0, 255, 255),
    2
)

cv2.imshow("Plant Disease Detection", img)

cv2.waitKey(5000)
cv2.destroyAllWindows()

esp.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
Disease Name : Tomato___healthy
Confidence Score : 99.99%
Healthy plant → Water


: 